# Mini-Project 2: Solar Micro-Grid Dispatch Planner

A rural health centre in Kasese has two demand constraints:

\[
3x + 2y = D_1, \qquad 4x + y = D_2
\]

Here, `x` is solar energy and `y` is battery energy, both in kWh. The CSV contains synthetic illustrative data, not real measurements.

## 1. Imports and load the 30-day CSV

In [ ]:
from pathlib import Path
import sys, timeit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path("/mnt/data/solar_microgrid_project")
sys.path.insert(0, str(ROOT))
from src.microgrid import MicroGrid, read_demands_csv, dispatch_report, descriptive_stats
from scipy.linalg import solve

demands = read_demands_csv(str(ROOT / "data" / "daily_demand.csv"))
display(demands.head())
print("Days:", len(demands))

## 2. Determinant and condition number
A non-zero determinant means a unique exact solution exists. The condition number measures sensitivity to changes in the input; it does not tell us whether the solution is physically feasible.

In [ ]:
grid = MicroGrid()
print("Matrix:\n", grid.matrix)
print("Determinant:", grid.determinant)
print("Condition number:", grid.condition_number)

The determinant is `3×1 − 2×4 = −5`, which is non-zero. Therefore, the coefficient matrix is invertible and there is a unique mathematical solution for any right-hand-side demand vector.

## 3. Interactive input with validation
The following function rejects empty, non-numeric, negative and non-finite values. Uncomment the last three lines to enter a day interactively.

In [ ]:
def ask_nonnegative_number(prompt: str) -> float:
    while True:
        raw = input(prompt).strip()
        if not raw:
            print("Input cannot be empty.")
            continue
        try:
            value = float(raw)
        except ValueError:
            print("Please enter a number.")
            continue
        if not np.isfinite(value) or value < 0:
            print("Enter a finite, non-negative number.")
            continue
        return value

# d1 = ask_nonnegative_number("D1 daytime load (kWh): ")
# d2 = ask_nonnegative_number("D2 critical-equipment load (kWh): ")
# print("Solar x, battery y:", grid.solve_day(d1, d2))

## 4. Solve by loop and vectorisation
The right-hand-side matrix has shape 2×30. A vectorised call solves all days in one call.

In [ ]:
B = demands[["d1_kwh", "d2_kwh"]].to_numpy().T
def loop_solve():
    return np.column_stack([solve(grid.matrix, B[:, i]) for i in range(B.shape[1])])
def vector_solve():
    return solve(grid.matrix, B)

a, b = loop_solve(), vector_solve()
print("B shape:", B.shape, "result shape:", b.shape)
print("Equivalent results:", np.allclose(a, b))
print("Loop time for 1,000 runs:", timeit.timeit(loop_solve, number=1000))
print("Vectorised time for 1,000 runs:", timeit.timeit(vector_solve, number=1000))

Timing differences depend on the computer and may be small for only 30 days. Vectorisation avoids Python-level looping and is often more useful for larger batches.

## 5. Physical feasibility and NNLS
Negative energy is impossible. Such days are flagged and handled using non-negative least squares. NNLS may not satisfy both equations exactly, so the residual norm is reported.

In [ ]:
results = dispatch_report(demands, grid)
display(results.head(10))
print("Infeasible raw solutions:", int(results["infeasible_raw_solution"].sum()))
display(results["handling"].value_counts())

## 6. Statistics and volatility
The statistics module reports sample variance and standard deviation. Standard deviation has the same units (kWh) as energy and is easier to interpret as absolute variability.

In [ ]:
for col, label in [("solar_kwh", "Solar"), ("battery_kwh", "Battery")]:
    print(label, descriptive_stats(results[col]))
s_sd = descriptive_stats(results["solar_kwh"])["std_dev"]
b_sd = descriptive_stats(results["battery_kwh"])["std_dev"]
print("More volatile by standard deviation:", "Solar" if s_sd > b_sd else "Battery")

## 7. Cost model
The assignment gives illustrative rates of UGX 150/kWh for solar and UGX 450/kWh for battery energy.

In [ ]:
results["daily_cost_ugx"] = results["solar_kwh"]*150 + results["battery_kwh"]*450
print(f"Total 30-day cost: UGX {results['daily_cost_ugx'].sum():,.2f}")
print(f"Mean daily cost: UGX {results['daily_cost_ugx'].mean():,.2f}")

## 8. Stacked bar chart and daily cost

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
days = results["day"]
ax1.bar(days, results["solar_kwh"], label="Solar (kWh)")
ax1.bar(days, results["battery_kwh"], bottom=results["solar_kwh"], label="Battery (kWh)")
ax1.set_xlabel("Day")
ax1.set_ylabel("Energy dispatched (kWh)")
ax1.set_title("Daily Solar and Battery Dispatch with Daily Cost")
ax1.set_xticks(days)
ax1.tick_params(axis="x", rotation=45)
ax2 = ax1.twinx()
ax2.plot(days, results["daily_cost_ugx"], marker="o", label="Daily cost (UGX)")
ax2.set_ylabel("Daily cost (UGX)")
h1,l1 = ax1.get_legend_handles_labels()
h2,l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc="upper left")
fig.tight_layout()
plt.show()

## 9. Sensitivity analysis (1,000 draws)
Each demand is perturbed independently by ±5%. This shows how solar and battery solutions respond for the first day in the dataset.

In [ ]:
rng = np.random.default_rng(2026)
perturb = rng.uniform(0.95, 1.05, size=(2, 1000))
draws = B[:, :1] * perturb
samples = solve(grid.matrix, draws)
base_solution = solve(grid.matrix, B[:, :1])[:, 0]
for i, name in enumerate(["Solar", "Battery"]):
    print(name, "base:", base_solution[i], "min:", samples[i].min(),
          "max:", samples[i].max(), "sample SD:", samples[i].std(ddof=1))

## 10. Findings & Limitations

The determinant is −5, so the coefficient matrix has a unique mathematical solution. The condition number describes numerical sensitivity to small changes in demand inputs, but it does not establish whether the resulting energy quantities are physically possible. Loop-based and vectorised solutions should agree; timing differences depend on the computer and may be small for a 30-day dataset.

Some unconstrained solutions may return negative solar or battery quantities. These are physically infeasible, so the project flags them and uses non-negative least squares (NNLS). NNLS finds a non-negative approximation, but it may not satisfy both equations exactly; the residual norm is reported. The cost model applies the assignment's illustrative rates of UGX 150/kWh for solar and UGX 450/kWh for battery energy.

The data are synthetic, generated with a fixed random seed and an illustrative weekly pattern plus noise. They demonstrate reproducible analysis but cannot establish the real demand, reliability, cost or equipment needs of a health centre in Kasese. A real deployment would require measured load data and engineering validation of the equations, battery efficiency, equipment limits and state of charge.

## 11. Tests
From a terminal opened in the project folder, run `pytest -q`. The test suite checks normal solutions, negative input, singular matrices, empty statistics, and reproducible CSV generation.